In [2]:
import pandas as pd
import numpy as np
import sys

# Kütüphane kurulumu
!{sys.executable} -m pip install -q scikit-learn

# CSV yolunu ayarlama
csv_path = "Police_Transparency_-_Arrests_-_All_Data_(main_table___denormalized).csv"

try:
    df = pd.read_csv(csv_path, low_memory=False)
    print("✔ Veri seti başarıyla yüklendi.")
except Exception as e:
    print(f"❌ Veri yükleme hatası: {e}")
    raise SystemExit

print(f"Veri boyutu: {df.shape}")
display(df.head())


✔ Veri seti başarıyla yüklendi.
Veri boyutu: (47444, 40)


,X,Y,rin,primary_key,arrest_type,arrest_translation,arrest_dt,arrest_time,arrest_hour_of_day,location,...,severity_code,severity_trans,ChargeClassTranslation,ChargeGrouping,arrestee_sex,arrestee_race,arrestee_ethnicity,arrestee_genderTranslation,arrestee_RaceAndEthnicity,ESRI_OID
0,693985,882475,110528,TE20226631,O,On-View: Arrested when first observed/investi...,2022/09/24 21:17:00+00,2117,21,2XX E 5TH ST,...,NaN,Unknown,Arizona Revised Statutes,Miscellaneous,M,B,N,Male,Black or African American,3715748
1,710663,880237,110739,TE20226842,T,Taken Into Custody: Arrest on warrant or PC f...,2022/10/04 17:30:00+00,1730,17,9XX S ACORN AVE,...,F,Felony,Arizona Revised Statutes,Drug charges,M,W,N,Male,White,3715749
2,704259,878441,110587,TE20226690,T,Taken Into Custody: Arrest on warrant or PC f...,2022/09/27 21:52:00+00,2152,21,1XXX E APACHE BLVD,...,M,Misdemeanor,Arizona Revised Statutes,Assault & related charges,M,W,H,Male,Hispanic or Latino,3715750
3,693160,868321,110521,TE20226624,O,On-View: Arrested when first observed/investi...,2022/09/24 21:00:00+00,2100,21,4XXX S MILL AVE,...,F,Felony,Arizona Revised Statutes,Miscellaneous,M,B,N,Male,Black or African American,3715751
4,708569,883842,110789,TE20226892,T,Taken Into Custody: Arrest on warrant or PC f...,2022/10/07 03:54:00+00,354,3,2XXX W RIO SALADO PKWY,...,M,Misdemeanor,Arizona Revised Statutes,Escape & related charges,M,W,H,Male,Hispanic or Latino,3715752


In [3]:
# Yinelenen satırları kaldırma
print("\n--- TEKİLLEŞTİRME ---")

initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
removed = initial_rows - df.shape[0]

print(f"✔ {removed} adet yinelenen satır silindi.")
print(f"Yeni veri boyutu: {df.shape}")



--- TEKİLLEŞTİRME ---
✔ 0 adet yinelenen satır silindi.
Yeni veri boyutu: (47444, 40)


In [4]:
# Gereksiz sütunları kaldırma
columns_to_drop = [
    'X','Y','rin','primary_key','charge_rin','pin','ESRI_OID',
    'x_coordinate','y_coordinate',
    'arrest_officer',
    'ofc_age_range','ofc_genderTranslation','ofc_RaceAndEthnicity',
    'statute','class','severity_code'
]

existing_cols = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=existing_cols, inplace=True)

print(f"✔ {len(existing_cols)} sütun kaldırıldı.")
print("Kaldırılanlar:", existing_cols)


✔ 16 sütun kaldırıldı.
Kaldırılanlar: ['X', 'Y', 'rin', 'primary_key', 'charge_rin', 'pin', 'ESRI_OID', 'x_coordinate', 'y_coordinate', 'arrest_officer', 'ofc_age_range', 'ofc_genderTranslation', 'ofc_RaceAndEthnicity', 'statute', 'class', 'severity_code']


In [5]:
# Eksik değerleri inceleme
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
display(missing.head(20))
print("\nVeri boyutu:", df.shape)


area_name                     5.667735
zipcode                       5.225107
arrest_type                   1.199309
arrest_translation            1.199309
ChargeGrouping                0.265576
arrestee_ethnicity            0.073771
arrestee_age_range            0.067448
arrestee_race                 0.040047
arrestee_RaceAndEthnicity     0.040047
arrestee_genderTranslation    0.040047
arrestee_sex                  0.040047
charge_count                  0.002108
arrest_time                   0.000000
arrest_dt                     0.000000
location                      0.000000
arrest_hour_of_day            0.000000
jurisdiction                  0.000000
municipality                  0.000000
zone                          0.000000
grid                          0.000000
dtype: float64


Veri boyutu: (47444, 24)


In [6]:
print("\n--- Aykırı Değer İşleme ---")

if "charge_count" in df.columns:
    Q1 = df['charge_count'].quantile(0.25)
    Q3 = df['charge_count'].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5*IQR

    anomalies = df[df['charge_count'] > upper].shape[0]

    df.loc[df['charge_count'] > upper, 'charge_count'] = np.nan
    print(f"✔ {anomalies} adet aykırı değer NaN yapıldı.")
else:
    print("⚠ 'charge_count' bulunamadı.")



--- Aykırı Değer İşleme ---
✔ 856 adet aykırı değer NaN yapıldı.


In [7]:
# Düşük kategorili verileri tespit etme
categorical_cols = df.select_dtypes(include=['object','category','bool']).columns

threshold = 0.001
print(f"\n--- Düşük Frekanslı Kategoriler (%{threshold*100}) ---")

for col in categorical_cols:
    vc = df[col].value_counts(normalize=True)
    low = vc[vc < threshold].index.tolist()
    if low:
        print(f"• {col}: {len(low)} düşük frekanslı kategori → {low[:3]}...")



--- Düşük Frekanslı Kategoriler (%0.1) ---
• arrest_type: 1 düşük frekanslı kategori → ['S']...
• arrest_translation: 1 düşük frekanslı kategori → ['Summoned/Cited: Ticket issued, or long form charges approved, not taken to jail']...
• arrest_dt: 20549 düşük frekanslı kategori → ['2023/02/11 20:00:00+00', '2024/05/09 16:10:00+00', '2023/06/02 10:14:00+00']...
• location: 2015 düşük frekanslı kategori → [' PRIEST DR / W SOUTHERN AVE  ', '4XX W BROADWAY RD    ', '1XXX E VISTA DEL CERRO DR     ']...
• municipality: 2 düşük frekanslı kategori → ['SR        ', 'CH        ']...
• district: 2 düşük frekanslı kategori → ['UI    ', 'DT    ']...
• zone: 2 düşük frekanslı kategori → ['UI    ', '1     ']...
• grid: 223 düşük frekanslı kategori → ['2311  ', '0606  ', '1317  ']...
• charge: 5889 düşük frekanslı kategori → ['PRESCRIPT DRUG-POSSESS/USE                                                                                                                                                       

In [8]:
print("\n--- İmputasyon Başladı (Uyarısız Güvenli Versiyon) ---")

# SAYISAL İMPUTASYON (MOD)
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    if df[col].isna().any():
        mode_val = df[col].mode(dropna=True)[0]
        df[col] = df[col].fillna(mode_val)
        print(f"✔ '{col}' mod ({mode_val}) ile dolduruldu.")

# KATEGORİK İMPUTASYON (DAĞILIM KORUMA)
cat_cols = df.select_dtypes(include=['object','category','bool']).columns

def impute_with_dist(col):
    nonnull = df[col].dropna()
    if nonnull.empty:
        return  
    probs = nonnull.value_counts(normalize=True)
    nan_mask = df[col].isna()
    nan_count = nan_mask.sum()
    if nan_count == 0:
        return
    random_values = np.random.choice(probs.index, size=nan_count, p=probs.values)
    df.loc[nan_mask, col] = random_values
    print(f"✔ '{col}' kategorik imputasyon ile dolduruldu.")

for col in cat_cols:
    if df[col].isna().any():
        impute_with_dist(col)

print("\n✔ Tüm imputasyonlar uyarısız olarak tamamlandı.")



--- İmputasyon Başladı (Uyarısız Güvenli Versiyon) ---
✔ 'zipcode' mod (85281.0) ile dolduruldu.
✔ 'charge_count' mod (1.0) ile dolduruldu.
✔ 'arrest_type' kategorik imputasyon ile dolduruldu.
✔ 'arrest_translation' kategorik imputasyon ile dolduruldu.
✔ 'area_name' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_age_range' kategorik imputasyon ile dolduruldu.
✔ 'ChargeGrouping' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_sex' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_race' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_ethnicity' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_genderTranslation' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_RaceAndEthnicity' kategorik imputasyon ile dolduruldu.

✔ Tüm imputasyonlar uyarısız olarak tamamlandı.


In [9]:
print("\n--- Zaman Feature Engineering ---")

# Tutuklama tarihinden yıl, ay ve haftanın günü gibi yeni özellikler türetildi
if 'arrest_dt' in df.columns:
    df['arrest_dt'] = pd.to_datetime(df['arrest_dt'].astype(str).str.split().str[0], errors='coerce')
    df['arrest_year'] = df['arrest_dt'].dt.year
    df['arrest_month'] = df['arrest_dt'].dt.month
    df['arrest_day_of_week'] = df['arrest_dt'].dt.dayofweek

    # Yeni bir kategorik özellik oluşturuldu
    def get_time_cat(h):
        if pd.isna(h): return 'Bilinmiyor'
        if 0<=h<=6: return 'Gece'
        if 7<=h<=11: return 'Sabah'
        if 12<=h<=16: return 'Ogle'
        if 17<=h<=20: return 'Ikindi'
        return 'Aksam'

    df['time_category'] = df['arrest_hour_of_day'].apply(get_time_cat)

    df.drop(columns=[c for c in ['arrest_dt','arrest_time'] if c in df.columns], inplace=True)
    print("✔ Zaman özellikleri eklendi.")



--- Zaman Feature Engineering ---
✔ Zaman özellikleri eklendi.


In [10]:
print("\n--- Hedef Değişken ---")

df['is_onview_arrest'] = np.where(df['arrest_type']=='O', 1, 0)
df.drop(columns=[c for c in ['arrest_type','arrest_translation'] if c in df.columns],
        inplace=True)

print("✔ 'is_onview_arrest' oluşturuldu.")



--- Hedef Değişken ---
✔ 'is_onview_arrest' oluşturuldu.


In [11]:
# One-Hot Encoding
# Kategorik veriler makine öğrenimi modellerinin anlayabilmesi için sayısal formata dönüştürüldü
ohe_cols = [c for c in df.select_dtypes(include=['object','category','bool'])
            if df[c].nunique() < 50]

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True, dtype=int)

print(f"✔ OHE tamamlandı. Yeni sütun sayısı: {df.shape[1]}")


✔ OHE tamamlandı. Yeni sütun sayısı: 112


In [12]:
# StandardScaler ile Ölçeklendirme
from sklearn.preprocessing import StandardScaler

scale_cols = [c for c in ['charge_count','arrest_hour_of_day'] if c in df.columns]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("✔ StandardScaler uygulandı.")


✔ StandardScaler uygulandı.


In [13]:
# Feature Selection
X = df.drop(columns=['is_onview_arrest'])
Y = df['is_onview_arrest']

high_card_cols = X.select_dtypes(include=['object','category']).columns
X_processed = X.drop(columns=high_card_cols)

from sklearn.feature_selection import SelectKBest, f_classif

# ANOVA F-testi (SelectKBest) kullanılarak 
#hedef değişkenle istatistiksel olarak en anlamlı ilişkiye sahip olan en iyi K adet özellik seçildi
K = min(150, X_processed.shape[1])
selector = SelectKBest(k=K, score_func=f_classif)
selector.fit(X_processed, Y)

scores = pd.Series(selector.scores_, index=X_processed.columns).dropna().sort_values(ascending=False)
selected_features = scores.index[:K].tolist()

print(f"✔ En iyi {K} özellik seçildi.")
display(scores.head()) 


✔ En iyi 108 özellik seçildi.


C:\Users\miray\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [2] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\miray\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


ChargeGrouping_Drug charges                 1005.608424
severity_trans_Misdemeanor                   992.250196
ChargeGrouping_Assault & related charges     907.554373
area_name_Papago/North Tempe                 787.723344
zone_11                                      636.037737
dtype: float64

In [14]:
# Final veri seti
# Feature Selectiondan sonra seçilen özellikler ve hedef değişken birleştirildi
X_final = X_processed[selected_features]
Y_final = Y

df_final = pd.concat([X_final, Y_final], axis=1)

print("\n✔ Final dataset hazır.")
print(df_final.shape)
display(df_final.head())



✔ Final dataset hazır.
(47444, 108)


,ChargeGrouping_Drug charges,severity_trans_Misdemeanor,ChargeGrouping_Assault & related charges,area_name_Papago/North Tempe,zone_11,ChargeGrouping_Miscellaneous,zipcode,ChargeGrouping_Traffic,municipality_ME,ChargeGrouping_Criminal damage to property,...,area_name_Apache,arrestee_RaceAndEthnicity_Native Hawaiian or Pacific Islander,ChargeGrouping_MARIJUANA,arrestee_ethnicity_I,municipality_GU,zone_GUAD,arrestee_RaceAndEthnicity_Unknown,arrestee_age_range_65+,ChargeGrouping_Prostitution,is_onview_arrest
0,0,0,0,0,0,1,85281.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1,0,0,0,0,0,85288.0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,1,1,0,0,0,85281.0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,85282.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,1,0,0,0,0,85281.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Modeli eğitir, tahmin yapar ve metrikleri hesaplayarak listeye ekler 
def evaluate_model_fs_step(model, X_train, X_test, y_train, y_test, model_name, fs_method, results_list):
    
    # Random Forest modellerini eğitme
    try:
        model.fit(X_train, y_train)
    except:
        # Dummy veya Naive Bayes gibi modelleri eğitme
        model.fit(X_train, y_train) 

    y_pred = model.predict(X_test) 
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else [0] * len(y_test) 

    # Metrikleri hesaplama
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    try:
        roc_auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        roc_auc = np.nan

    # Sonuç listesine ekleme
    results_list.append([
        f"{model_name}", 
        fs_method, 
        acc, prec, rec, f1, roc_auc
    ])
    
    print(f"\n>>> {model_name} ({fs_method})")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

# Veri seti ayarlama
X_train_final, X_test_final, Y_train_final, Y_test_final = train_test_split(
    X_final, Y_final, test_size=0.3, random_state=42, stratify=Y_final
)

# Sonuç listesi başlatılıyor 
new_results = []
FS_YONTEMI = "ANOVA (SelectKBest)"


In [16]:
print("\n\n===== MODELLER: NAIVE BAYES VE RANDOM FOREST (ANOVA FS SONRASI) =====\n")

# 1. DUMB MODEL (BASELINE)
print("--- 1. Dumb (Baseline) Model ---")
dummy_model_anova = DummyClassifier(strategy='most_frequent', random_state=42)
evaluate_model_fs_step(dummy_model_anova, X_train_final, X_test_final, Y_train_final, Y_test_final, "Dummy Classifier", FS_YONTEMI, new_results)

# 2. NAIVE BAYES SINIFLANDIRICI 
print("\n--- 2. Naive Bayes ---")
nb_model = GaussianNB()
evaluate_model_fs_step(nb_model, X_train_final, X_test_final, Y_train_final, Y_test_final, "Naive Bayes", FS_YONTEMI, new_results)

# 3. RANDOM FOREST SINIFLANDIRICI (HIPER-PARAMETRE AYARLAMA) 
print("\n--- 3. Random Forest ---")

# Sınıf Dengesizliği için class_weight='balanced' eklendi
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced') 

# Randomized Search (Rastgele Arama)
# Denenecek parametrelerin aralıkları
param_dist = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# En iyi parametreleri bulma
# 10 farklı parametre kombinasyonu 3 katlı çarpraz doğrulama ile denendi 
rf_random = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=param_dist, 
    n_iter=10, 
    cv=3, 
    scoring='f1', # Sınıf dengesizliği için F1 skorunu optimize ediyoruz
    random_state=42, 
    n_jobs=-1, 
    verbose=0 
)

# Modeli eğitme
rf_random.fit(X_train_final, Y_train_final)
best_rf_model = rf_random.best_estimator_

print(f"\nEn İyi Random Forest Parametreleri: {rf_random.best_params_}")

# En iyi modeli değerlendirme
evaluate_model_fs_step(best_rf_model, X_train_final, X_test_final, Y_train_final, Y_test_final, "Random Forest (Tuned)", FS_YONTEMI, new_results)

print("\n\n===== TÜM MODELLERİN KARŞILAŞTIRMA TABLOSU =====\n")
df_final_results = pd.DataFrame(new_results, columns=['Model', 'FS Yöntemi', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'])
display(df_final_results)



===== MODELLER: NAIVE BAYES VE RANDOM FOREST (ANOVA FS SONRASI) =====

--- 1. Dumb (Baseline) Model ---

>>> Dummy Classifier (ANOVA (SelectKBest))
Accuracy : 0.7588
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000
ROC-AUC  : 0.5000

--- 2. Naive Bayes ---

>>> Naive Bayes (ANOVA (SelectKBest))
Accuracy : 0.3074
Precision: 0.2552
Recall   : 0.9752
F1 Score : 0.4045
ROC-AUC  : 0.5631

--- 3. Random Forest ---

En İyi Random Forest Parametreleri: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': None, 'bootstrap': False}

>>> Random Forest (Tuned) (ANOVA (SelectKBest))
Accuracy : 0.8400
Precision: 0.6717
Recall   : 0.6580
F1 Score : 0.6648
ROC-AUC  : 0.8730


===== TÜM MODELLERİN KARŞILAŞTIRMA TABLOSU =====



,Model,FS Yöntemi,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dummy Classifier,ANOVA (SelectKBest),0.758817,0.000000,0.000000,0.000000,0.500000
1,Naive Bayes,ANOVA (SelectKBest),0.307433,0.255163,0.975240,0.404494,0.563134
2,Random Forest (Tuned),ANOVA (SelectKBest),0.839961,0.671722,0.658025,0.664803,0.872951
